# Purpose:
- Generate czstack id - hcr id matching table from the last landmarks file

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

In [4]:
def get_czstack_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[0][2:])
    else:
        return -1


def get_hcr_id(x):
    if x.startswith('cz'):
        return int(x.split('-')[1][3:])
    else:
        return -1

In [3]:
mouse_id = 767018
save_dir = Path(f'/root/capsule/scratch/{mouse_id}_coreg')
qced_files = save_dir.glob(f'{mouse_id}_landmarks_matched_*_qced.csv')
max_iternums = np.max([int(fn.name.split('_iter')[1].split('_')[0]) for fn in qced_files])
print(max_iternums)


3


In [ ]:
for iternum in range(1, max_iternums+1):
    save_fn = save_dir / f'{mouse_id}_coreg_table_iter{iternum}.csv'
    if save_fn.exists():
        print(f'Coreg table from iter {iternum} for {mouse_id} exists. Skip...')
    else:
        filename = next(save_dir.glob(f'{mouse_id}_landmarks_matched_*iter{iternum}_*qced.csv'))
        landmarks = pd.read_csv(filename)
        columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
        landmarks.columns = columns


        landmarks['czstack_id'] = landmarks.ids.apply(get_czstack_id)
        landmarks['hcr_id'] = landmarks.ids.apply(get_hcr_id)
        landmarks = landmarks[landmarks.czstack_id != -1]
        landmarks = landmarks.query('active')
        landmarks = landmarks.sort_values(by='czstack_id')[['czstack_id', 'hcr_id']].reset_index(drop=True)
        landmarks.to_csv(save_fn, index=False)

In [11]:
landmarks

,czstack_id,hcr_id
0,2,9906
1,3,9874
2,7,5957
3,8,7239
4,12,6407
...,...,...
565,676,103596
566,677,105352
567,678,102360
568,682,104705
